# Part 3: ResNet-18 Permutation Hardness Analysis

This notebook reuses the Part 1 ResNet-18 baseline results and saved tile permutations to compute permutation hardness metrics without retraining.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

In [ ]:
current = Path.cwd().resolve()
for candidate in [current, *current.parents]:
    if (candidate / 'src').is_dir() and (candidate / 'requirements.txt').exists():
        ROOT = candidate
        break
else:
    ROOT = current

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

In [ ]:
def install_project_requirements_for_colab(project_root: Path) -> None:
    """Install non-PyTorch dependencies in Colab without replacing CUDA-matched torch wheels."""
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return

    import subprocess

    requirements_path = project_root / 'requirements.txt'
    filtered_requirements = Path('/tmp/mlds_colab_requirements.txt')
    skip_prefixes = ('torch', 'torchvision')
    filtered_lines = [
        line
        for line in requirements_path.read_text().splitlines()
        if not line.strip().lower().startswith(skip_prefixes)
    ]
    filtered_requirements.write_text('\n'.join(filtered_lines) + '\n')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(filtered_requirements)])


In [ ]:
install_project_requirements_for_colab(ROOT)

ROOT

In [ ]:

import pandas as pd
from IPython.display import Image, display

import src.evaluation.experiment_results as experiment_results
from src.utils.config import Part3ExperimentConfig

experiment_results = importlib.reload(experiment_results)
load_part1_model_results = experiment_results.load_part1_model_results
load_part3_results = experiment_results.load_part3_results
part3_output_paths = experiment_results.part3_output_paths
run_part3_hardness_analysis = experiment_results.run_part3_hardness_analysis



## Configuration
Define Part 3 settings in `Part3ExperimentConfig`, reusing the Part 1 model/data/permutation outputs.


In [ ]:

config = Part3ExperimentConfig()
results_dir = Path(config.results_dir)
figures_dir = Path(config.figures_dir)
part1_results_csv = results_dir / 'part1_raw_results.csv'
permutation_csv = results_dir / 'part1_permutations.csv'
output_paths = part3_output_paths(str(results_dir), str(figures_dir))

pd.DataFrame([
    {
        'part': config.part,
        'config_name': config.config_name,
        'model_name': config.model_name,
        'grid_sizes': config.grid_sizes,
        'num_permutations': config.num_permutations,
        'seed': config.seed,
        'alpha_center': config.alpha_center,
        'weight_center': config.weight_center,
        'weight_dist': config.weight_dist,
    }
])


In [ ]:

METRIC_LABELS = {
    'global_tile_displacement': 'Global displacement',
    'center_weighted_displacement': 'Center-weighted displacement',
    'combined_hardness_score': 'Combined hardness',
}
METRIC_COLUMNS = list(METRIC_LABELS)


def display_part3_metrics_table(metrics: pd.DataFrame, max_rows: int = 12):
    table = metrics.copy()
    table['_permuted_first'] = ~((table['grid_size'] == 1) | (table['permutation_id'] == 0))
    table = table.sort_values(['_permuted_first', 'grid_size', 'permutation_id'], ascending=[False, True, True])
    table = table.drop(columns=['_permuted_first'])
    table = table.rename(columns=METRIC_LABELS | {
        'grid_size': 'Grid',
        'num_tiles': 'Tiles',
        'permutation_id': 'Permutation',
        'permutation_seed': 'Seed',
    })
    display(table.head(max_rows).style.format(precision=3))


def display_part3_joined_table(joined: pd.DataFrame, max_rows: int = 12):
    columns = ['grid_size', 'num_tiles', 'permutation_id', 'best_val_accuracy', *METRIC_COLUMNS]
    table = joined.loc[:, [column for column in columns if column in joined.columns]].copy()
    table['_permuted_first'] = ~((table['grid_size'] == 1) | (table['permutation_id'] == 0))
    table = table.sort_values(['_permuted_first', 'grid_size', 'permutation_id'], ascending=[False, True, True])
    table = table.drop(columns=['_permuted_first'])
    table = table.rename(columns=METRIC_LABELS | {
        'grid_size': 'Grid',
        'num_tiles': 'Tiles',
        'permutation_id': 'Permutation',
        'best_val_accuracy': 'Best validation accuracy',
    })
    display(table.head(max_rows).style.format(precision=3))


def display_part3_correlations_table(correlations: pd.DataFrame):
    table = correlations.copy()
    table['metric'] = table['metric'].replace(METRIC_LABELS)
    table = table.rename(columns={
        'group': 'Group',
        'metric': 'Metric',
        'pearson': 'Pearson',
        'spearman': 'Spearman',
        'n': 'N',
    })
    display(table.style.format({'Pearson': '{:.3f}', 'Spearman': '{:.3f}'}))


## Data Loading
Load existing Part 1 results filtered to the configured Part 3 model.

In [ ]:

part1_model_results = load_part1_model_results(str(part1_results_csv), config.model_name)
print(f'Part 1 {config.model_name} result rows: {len(part1_model_results)}')
display(
    part1_model_results[
        ['grid_size', 'num_tiles', 'permutation_id', 'best_val_accuracy']
    ].sort_values(['grid_size', 'permutation_id']).head(12).style.format(precision=3)
)


## Compute Hardness Metrics
Compute permutation-only metrics, join them with ResNet-18 accuracy, and save outputs.

In [ ]:

analysis_results = run_part3_hardness_analysis(
    results_dir=str(results_dir),
    figures_dir=str(figures_dir),
    part1_results_csv=str(part1_results_csv),
    permutation_csv=str(permutation_csv),
    grid_sizes=config.grid_sizes,
    num_permutations=config.num_permutations,
    seed=config.seed,
    alpha_center=config.alpha_center,
    weight_center=config.weight_center,
    weight_dist=config.weight_dist,
    model_name=config.model_name,
    verbose=True,
    show_progress=True,
)

display_part3_metrics_table(analysis_results['metrics'])
display_part3_joined_table(analysis_results['joined'])


## Correlations
Display Pearson and Spearman correlations between each hardness metric and the configured model validation accuracy.

In [ ]:
saved_results = load_part3_results(str(results_dir))
display_part3_correlations_table(saved_results['correlations'])

if saved_results['correlations'][['pearson', 'spearman']].isna().any().any():
    print(
        'NaN means the correlation is undefined because the selected accuracy '
        'or metric values are constant in the joined Part 3 data.'
    )


## Metric Plots
Display saved metric-vs-accuracy plots.

In [ ]:
output_paths = part3_output_paths(str(results_dir), str(figures_dir))
for figure_path in output_paths['plots']:
    display(Image(filename=figure_path))
if not output_paths['plots']:
    print('No Part 3 plots found yet.')